In [38]:
import torch
import pandas as pd
import numpy as np
import logging
import argparse
import transformers
import re
from nltk.tokenize import sent_tokenize
from transformers import BertTokenizer, BertForMaskedLM
import math
import matplotlib.pyplot as plt
import string
import gc

In [ ]:
import pandas as pd

In [39]:
from tqdm.notebook import tqdm

In [40]:
import sys
import os

In [41]:
sys.path.append(os.path.dirname(os.getcwd()))

In [42]:
from utils.load_word_list import WordLoader
from utils import wordlist_v3

In [44]:
# a logger for both screen output and file output
logging.basicConfig(level=logging.INFO,
                    format='%(message)s',
                    handlers=[logging.FileHandler('../output.txt', 'w'),
                              logging.StreamHandler()])
logger = logging.getLogger()

In [45]:
logger.info("Logger initialized.")

Logger initialized.


# Main Function code

## Main 1

In [46]:
def load_base_model_and_tokenizer(base_model_name):
    """
    Load a pretrained model and its corresponding tokenizer.
    
    :param base_model_name: str, the name of the pretrained model.
    :return: the model and tokenizer instances.
    """
    print(f"Loading base model {base_model_name}...")
    base_model = transformers.AutoModelForMaskedLM.from_pretrained(base_model_name)
    base_tokenizer = transformers.AutoTokenizer.from_pretrained(base_model_name)
    
    # Switch model to evaluation mode
    base_model.eval()
    
    return base_model, base_tokenizer


prepositions = set(['a', 'an', 'the', 'then', 'this', 'these', 'that', 'those', 'which', 'what', 'where', 'however',  
    "aboard", "about", "above", "across", "after", "against", "along", "amid", "among",
    "around", "as", "at", "before", "behind", "below", "beneath", "beside", "between",
    "beyond", "by", "down", "during", "for", "from", "in", "inside", "into", "like",
    "near", "of", "off", "on", "onto", "out", "outside", "over", "past", "since",
    "through", "to", "toward", "under", "underneath", "until", "up", "upon", "with",
    "within", "without"
])


def create_masked_sentences_and_words(sentences, tokenizer):
    """
    Create masked versions of each sentence and extract the masked word.
    
    For each word in a sentence, this function masks the word (i.e., replaces it with a special mask token), 
    and then appends the modified sentence to a list. It also keeps track of which word was masked for each 
    modified sentence.

    :param sentences: list of str, the input sentences.
    :param tokenizer: the tokenizer instance.
    :return: list of masked sentences and corresponding masked words.
    """
    def is_number(s):
        """Check if the input string s is a number (int or float)."""
        if s.isdigit():
            return True
        try:
            float(s)
            return True
        except ValueError:
            return False
    

    # Initialize lists to store masked versions of sentences and the words that were masked.
    masked_sentences = []
    masked_words = []

    # Iterate through each sentence provided in the input.
    for sentence in sentences:
        # Tokenize the sentence, i.e., split the sentence into individual words or subwords depending on the tokenizer.
        words = tokenizer.tokenize(sentence)
        
        # Iterate through each word (or subword token) in the tokenized sentence.
        for i, word in enumerate(words):
            
            if (word.strip(string.punctuation) == "" or word.lower() in prepositions or is_number(word)):
                continue
            
            # Create a masked version of the sentence. 
            # This is done by replacing the current word (denoted by index i) with the mask token.
            # words[:i] gives all the words before the current word.
            # words[i+1:] gives all the words after the current word.
            # [tokenizer.mask_token] is a list containing the special mask token.
            # These pieces are concatenated together to form the masked sentence.
            masked_sentence = tokenizer.convert_tokens_to_string(words[:i] + [tokenizer.mask_token] + words[i+1:])
            
            # Append the masked sentence to the list of masked sentences.
            masked_sentences.append(masked_sentence)
            
            # Append the current word (the word that was masked) to the list of masked words.
            masked_words.append(word)

    # Return the lists containing the masked versions of sentences and the words that were masked in each of them.
    return masked_sentences, masked_words



In [47]:
def analyze_text(paragraph, word_dict, model, tokenizer, args):
    def split_into_batches(data, batch_size):
        return [data[i:i + batch_size] for i in range(0, len(data), batch_size)]

    def tokenize_and_find_mask_tokens(sentence_batch, tokenizer, device):
        inputs = tokenizer(sentence_batch, return_tensors="pt", padding=True, truncation=True, max_length=512)
        mask_token_indices = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)
        inputs = {key: value.to(device) for key, value in inputs.items()}
        return inputs, mask_token_indices

    # Basic checks before processing
    if not paragraph:
        logging.warning("Input paragraph is empty. Nothing to analyze.")
        return
    if not word_dict:
        logging.warning("No given_words provided. Analysis will be skipped.")
        return
    if not tokenizer:
        logging.warning("Tokenizer is not provided or initialized properly.")
        return
    if not model:
        logging.warning("Model is not provided or initialized properly.")
        return
    
    
    sentences = paragraph
    # sentences = split_paragraph_to_sentences(paragraph)
    #sentences = sentences[1:-1]
    #sentences = [''.join(sentences)]
    
    stats_whole_paragraph = {}
    stats_whole_paragraph['word'] = []
    for category, gender_data in word_dict.items():
        stats_whole_paragraph[category] = {}
        for gender, words_id in gender_data.items():
            stats_whole_paragraph[category][gender] = []
    
    
    # This is a list of masked sentences, the length should # of tokens
    masked_sentences, masked_words = create_masked_sentences_and_words(sentences, tokenizer)

    masked_sentence_batches = split_into_batches(masked_sentences, args.batch_size)
    masked_words_batches = split_into_batches(masked_words, args.batch_size)
    
    # Process each batch of sentences.
    for sentence_batch, masked_word_batch in zip(masked_sentence_batches, masked_words_batches):
        inputs, mask_token_indices = tokenize_and_find_mask_tokens(sentence_batch, tokenizer, args.device)
        
        # `predictions` is a tensor with shape (batch_size, sequence_length, vocab_size) containing softmax probabilities.
        with torch.no_grad():
            outputs = model(**inputs)
            softmax_scores = torch.nn.functional.softmax(outputs.logits, dim=-1)
            softmax_scores = softmax_scores.cpu()
            softmax_scores = softmax_scores.numpy()
        
        for sentence_idx in range(len(sentence_batch)):
            # Get the mask token indices for the current sentence.
            # This extracts the positions in the sentence where the mask token appears.
            sentence_mask_token_indices = mask_token_indices[1][mask_token_indices[0] == sentence_idx]

            # The sentence_mask_token_indices identifies the location of the masked token within the sentence.
            assert len(sentence_mask_token_indices) == 1, f"Expected only one mask token index for ONE sentence, but found {len(sentence_mask_token_indices)}."
            
            softmax_scores_per_sentence = softmax_scores[sentence_idx, sentence_mask_token_indices[0], :]
            
            masked_word = masked_word_batch[sentence_idx]
            original_word_id = tokenizer.convert_tokens_to_ids(masked_word)
            p_mask = softmax_scores_per_sentence[original_word_id]
                        
            for key, gender_data in word_dict.items():
                for direction, words_id in gender_data.items():
                    word_scores = softmax_scores_per_sentence[words_id]
                    stats_whole_paragraph[key][direction].append(word_scores)
        
            stats_whole_paragraph['word'].append( (masked_word, p_mask) )
        
        del softmax_scores, inputs
        torch.cuda.empty_cache()
    
    for key, gender_data in word_dict.items():
        for direction, words_id in gender_data.items():
            stats_whole_paragraph[key][direction] = np.stack(stats_whole_paragraph[key][direction])
    
    torch.cuda.empty_cache()
    gc.collect()
    return stats_whole_paragraph

## Main calcualtion

In [49]:
def output_given_words_probability_and_stats(predictions, tokenizer, masked_word, all_words_data, mask_token_index, sentence_idx):
    softmax_scores = predictions[sentence_idx, mask_token_index, :]
    
    original_word_id = tokenizer.convert_tokens_to_ids(masked_word)
    p_mask = softmax_scores[original_word_id].item()

    stats = {}
    epsilon = 1e-8 

    for key, word_data in all_words_data.items():
        stats[key] = {}
        for direction, word_list in word_data.items():
            word_ids = tokenizer.convert_tokens_to_ids(word_list)
            word_scores = softmax_scores[word_ids]

            # Filter out scores below threshold
            valid_indices = word_scores > epsilon
            valid_word_scores = word_scores[valid_indices]
            valid_word_list = np.array(word_list)[valid_indices.cpu().numpy()]

            if valid_word_scores.numel() == 0:  # If no valid scores, skip this group
                continue

            stats[key][direction] = {
                'word_scores': dict(zip(valid_word_list, valid_word_scores.tolist())),
                'original_word_score': {masked_word: p_mask}
                
            }

    return stats

In [52]:
def compute_word_scores(predictions, tokenizer, word_list):
    word_ids = torch.tensor(tokenizer.convert_tokens_to_ids(word_list))
    word_scores = predictions[word_ids]
    return [(word, score.item()) for word, score in zip(word_list, word_scores)]

def filter_and_rank_scores(all_word_scores, alpha=100):
    # Sort and rank words based on their scores
    sorted_scores = sorted(all_word_scores, key=lambda x: x[1], reverse=True)
    total_words = len(sorted_scores)
    alpha_threshold = int(np.ceil(total_words * (alpha / 100.0)))  # Determine the rank threshold for alpha quantile
    filtered_scores = sorted_scores[:alpha_threshold]  # Keep only scores within the alpha to 100th percentile
    
    # Generate rankings for the filtered set of words
    rankings = {word: rank + 1 for rank, (word, _) in enumerate(filtered_scores)}
    return rankings, filtered_scores

def compute_mean_ranking(rankings, word_list):
    ranks = [rankings.get(word) for word in word_list if word in rankings]  # Consider only ranked (filtered-in) words
    mean_rank = sum(ranks) / len(ranks) if ranks else None  # Return None if no words from the list were ranked
    return mean_rank

def output_given_words_probability_and_stats(predictions, tokenizer, masked_word, all_words_data, mask_token_index, sentence_idx, alpha=10):
    softmax_scores = torch.nn.functional.softmax(predictions[sentence_idx, mask_token_index, :], dim=-1).cpu()
    original_word_id = torch.tensor([tokenizer.convert_tokens_to_ids(masked_word)])
    p_mask = softmax_scores[original_word_id].item()

    stats = {}

    for key, word_data in all_words_data.items():
        combined_word_scores = []
        for direction, word_list in word_data.items():
            combined_word_scores.extend(compute_word_scores(softmax_scores, tokenizer, word_list))
        
        # Filter rankings based on the alpha quantile and get rankings for filtered words
        combined_rankings, filtered_scores = filter_and_rank_scores(combined_word_scores, alpha)
        
        stats[key] = {}
        for direction, word_list in word_data.items():
            # Determine which words in the current direction were included in the filtered set
            filtered_direction_words = [ word for word, score in filtered_scores if word in word_list]
            filtered_direction_scores = [ score for word, score in filtered_scores if word in word_list]
            direction_rankings = {word: combined_rankings[word] for word in filtered_direction_words}
            mean_ranking = compute_mean_ranking(combined_rankings, filtered_direction_words)
            
            # Only include statistics for words that were within the specified quantile
            stats[key][direction] = {
                # 'word_scores': dict(zip(filtered_direction_words, filtered_direction_scores)),
                # 'original_word_score': {masked_word: p_mask},
                # 'rankings': direction_rankings,
                'weighted_prob_ratio': mean_ranking if mean_ranking is not None else 100,  # Handle case with no ranked words
            }

    return stats


# Main running code

## Args Parse

In [56]:
parser = argparse.ArgumentParser()
parser.add_argument("--device", default="cuda:0" if torch.cuda.is_available() else "cpu")
parser.add_argument("--batch_size", type=int, default=32)
parser.add_argument("--base_model_name", default="bert-base-uncased")
parser.add_argument("--topk", type=int, default=20, help="Specify the number of top-k outputs to be printed")
parser.add_argument("--print_or_log", default="log", help="log or print or None")
parser.add_argument("--output_masked_top_word", default=False)
parser.add_argument("--output_prob_each_word", default=False)
args = parser.parse_args('')


In [61]:
args

Namespace(base_model_name='bert-base-uncased', batch_size=32, device='cuda:0', output_masked_top_word=False, output_prob_each_word=False, print_or_log=None, topk=20)

## load MLM and tokenizer

In [62]:
device = args.device
base_model_name = args.base_model_name
batch_size = args.batch_size
topk = args.topk

model, tokenizer = load_base_model_and_tokenizer(base_model_name)
model = model.to(device)

Loading base model bert-base-uncased...


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


## load wordlist

In [64]:
# for version 2 word list
loader = WordLoader(wordlist_v3)
all_words_dict, _ = loader.load_all_words(subkeys=['psy', 'role', 'wfc', 'gsc'])

In [65]:
def filter_words(nested_dict, tokenizer):
    # Get the tokenizer's vocabulary
    vocab = tokenizer.get_vocab()
    
    # Initialize a new dictionary to store the valid words
    valid_words_dict = {}
    
    # Iterate over the keys and subkeys of the nested dictionary
    for key, sub_dict in nested_dict.items():
        valid_words_dict[key] = {}
        for sub_key, word_list in sub_dict.items():
            valid_words = []
            for word in word_list:
                # Check if the word is in the tokenizer's vocabulary
                if word in vocab:
                    valid_words.append(word)
                elif 'Ġ'+ word in vocab:
                    valid_words.append(word)
                else:
                    # If the word is not in the vocabulary, print the word and how it's subtokenized
                    subtokens = tokenizer.tokenize(word)
                    print(f"{word} is not in the tokenizer's vocabulary. Subtokenized as: {subtokens}")
            valid_words_dict[key][sub_key] = valid_words
    
    return valid_words_dict

In [ ]:
valid_words_dict = filter_words(all_words_dict, tokenizer)

In [67]:
valid_words_dict

{'psy': {'fem': ['accuracy',
   'accurately',
   'accurate',
   'attention',
   '##care',
   'caring',
   'collaboration',
   'collaborated',
   'collaborative',
   'collaborating',
   'collaborate',
   'committed',
   'creative',
   'dedicated',
   'details',
   'detailed',
   'detailing',
   'detail',
   'diplomatic',
   'following',
   'follow',
   'follows',
   'friendly',
   'organized',
   'patient',
   'polite',
   'thoughtful',
   'welcome',
   'affection',
   'child',
   'cheerful',
   'commit',
   'communal',
   'compassion',
   'connect',
   'consider',
   'cooperation',
   'dependent',
   'emotional',
   'feminine',
   'gentle',
   'honest',
   'kind',
   'kinship',
   'loyalty',
   'modest',
   'pleasant',
   'polite',
   'quiet',
   'responding',
   'sensitive',
   'supportive',
   'sympathetic',
   'tender',
   'together',
   'trust',
   'understanding',
   'warm',
   'yielding',
   'love',
   'shy',
   '##lli',
   'child',
   '##ten',
   '##mun',
   'court',
   '##su',


In [ ]:
import pickle
with open('./valid_word_dict.pkl', 'wb') as f:
    pickle.dump(valid_words_dict,f)

In [68]:
valid_word_ids = {}
for category, gender_data in valid_words_dict.items():
    valid_word_ids[category] = {}
    for gender, words in gender_data.items():
        # Convert tokens to IDs
        word_ids = tokenizer.convert_tokens_to_ids(words)
        # Store the result
        valid_word_ids[category][gender] = word_ids

In [70]:
def split_into_sentences(text):
    alphabets = "([a-z])"  # Changed to lowercase character range
    prefixes = "(mr|st|mrs|ms|dr)[.]"  # Changed to lowercase prefixes
    suffixes = "(inc|ltd|jr|sr|co)"  # Changed to lowercase suffixes
    starters = "(mr|mrs|ms|dr|he\s|she\s|it\s|they\s|their\s|our\s|we\s|but\s|however\s|that\s|this\s|wherever)"  # Changed to lowercase starters
    acronyms = "([a-z][.][a-z][.](?:[a-z][.])?)"  # Changed to lowercase acronyms
    websites = "[.](com|net|org|io|gov)"  # Changed to lowercase websites

    text = " " + text.lower() + "  "  # Convert input text to lowercase
    text = text.replace("\n", "<stop>")
    text = text.replace('*', '')
    text = re.sub(prefixes, "\\1<prd>", text)
    text = re.sub(websites, "<prd>\\1", text)
    if "ph.d" in text: text = text.replace("ph.d.", "ph<prd>d<prd>")
    text = re.sub("\s" + alphabets + "[.] ", " \\1<prd> ", text)
    text = re.sub(acronyms + " " + starters, "\\1<stop> \\2", text)
    text = re.sub(alphabets + "[.]" + alphabets + "[.]" + alphabets + "[.]", "\\1<prd>\\2<prd>\\3<prd>", text)
    text = re.sub(alphabets + "[.]" + alphabets + "[.]", "\\1<prd>\\2<prd>", text)
    text = re.sub(" " + suffixes + "[.] " + starters, " \\1<stop> \\2", text)
    text = re.sub(" " + suffixes + "[.]", " \\1<prd>", text)
    text = re.sub(" " + alphabets + "[.]", " \\1<prd>", text)
    if "”" in text: text = text.replace(".”", "”.")
    if "\"" in text: text = text.replace(".\"", "\".")
    if "!" in text: text = text.replace("!\"", "\"!")
    if "?" in text: text = text.replace("?\"", "\"?")
    text = text.replace(".", ".<stop>")
    text = text.replace("?", "?<stop>")
    text = text.replace("!", "!<stop>")
    text = text.replace("<prd>", ".")
    sentences = text.split("<stop>")
    sentences = sentences[:-1]
    sentences = [re.sub(r'^[^a-z]+|[^a-z]+$', '', sentence) for sentence in sentences] # Remove leading and ending punctuation from each sentence
    sentences = [s.strip() for s in sentences if s.strip() and len(s.split()) > 5]  # longer than two words
    
    

    return sentences

## Store results

In [28]:
df_app = pd.read_csv('../linkdin_all_app_job.csv')

In [29]:
df_app.columns

Index(['JobApplication', 'JobText'], dtype='object')

In [30]:
df_app.shape

(33244, 2)

In [31]:
df_app

,JobApplication,JobText
0,Subject: Application for Hearing Care Provider...,Overview\n\nHearingLife is a national hearing ...
1,"[Your Name] \n[Your Address] \n[City, State,...",Metalcraft of Mayville\nMetalcraft of Mayville...
2,"[Your Name]\n[Your Address]\n[City, State, ZIP...",\nThe TSUBAKI name is synonymous with excellen...
3,"[Your Name] \n[Your Address] \n[City, State,...",descriptionTitle\n\n Looking for a great oppor...
4,"[Your Name] [Your Address] [City, State, Zip C...","Job Summary\nAt iHerb, we are on a mission to ..."
...,...,...
33239,Subject: Application for Marketing Manager Pos...,Are you a dynamic and creative marketing profe...
33240,"[Your Name]\n[Your Address]\n[City, State, Zip...","A fast-fashion wholesaler, is looking for a fu..."
33241,Subject: Job Application for DryerVentz DuctVe...,DuctVentz is a dryer and A/C – heat vent clean...
33242,Subject: Application for Insurance Agent Posit...,While many industries were hurt by the last fe...


In [76]:
#import pickle
import json
# Assuming the existence of necessary functions and imports
# Example: split_into_sentences, analyze_text


save_interval = 10000  # Save after every 10000 paragraphs
full_result = []
bad_list = []

last_check = 0
for row in tqdm(df_app.itertuples(), total=df_app.shape[0], desc="Processing"):
    i = row.Index
    try:
        filtered_sentences = split_into_sentences(row.JobText)
        stats = analyze_text(filtered_sentences, valid_word_ids, model, tokenizer, args)
        full_result.append((row.Index,stats))
    except Exception as e:
        print(f"Error processing paragraph {i}: {e}")
        bad_list.append({i:e})  # Optionally, keep track of paragraphs that caused errors
        continue
        

    # Save periodically
    if (row.Index + 1) % save_interval == 0:
        np.save(f'./Linkdin_results/results_app_partial_{row.Index - save_interval}_to_{row.Index}.npy', full_result) 
        print(f"Saved progress up to paragraph {i+1}")
        del full_result
        full_result = []
        last_check = save_interval+last_check
        
        torch.cuda.empty_cache()
        gc.collect()
        
        
    torch.cuda.empty_cache()
    #gc.collect()


# Optionally, save the indices of paragraphs that caused errors
if bad_list:
    with open('bad_list.json', 'w') as f:
        json.dump(bad_list, f)
    print("List of problematic paragraphs saved.")

Processing:   0%|          | 0/100 [00:00<?, ?it/s]

In [77]:
np.save(f'./label_results.npy', full_result)